In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score, average_precision_score, recall_score, precision_score
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectPercentile, mutual_info_classif, SelectFromModel, SelectKBest

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


In [2]:
def _make_ohe():
    # sklearn 新旧版本兼容：OneHotEncoder(sparse_output=...) vs (sparse=...)
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor(cols, num_cols, cat_cols) -> ColumnTransformer:
    """给定一组原始列名，构建对应的预处理器（数值/分类分别处理），输出可用于模型的矩阵。"""
    cols = list(cols)
    num_in = [c for c in cols if c in set(num_cols)]
    cat_in = [c for c in cols if c in set(cat_cols)]

    transformers = []
    if num_in:
        transformers.append(
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num_in)
        )
    if cat_in:
        transformers.append(
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", _make_ohe()),
            ]), cat_in)
        )

    if not transformers:
        raise ValueError("build_preprocessor(): 传入的 cols 里既没有数值列也没有分类列。")

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,  # 允许输出稀疏矩阵，适合 one-hot
    )


def get_score_vector(estimator: BaseEstimator, X):
    """返回用于 AUC 的连续得分：优先 decision_function，否则用 predict_proba[:,1]。"""
    if hasattr(estimator, "decision_function"):
        return estimator.decision_function(X)
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X)[:, 1]
    raise ValueError("模型既没有 decision_function 也没有 predict_proba，无法计算 AUC。")


def nested_cv_evaluate(
    X: pd.DataFrame,
    y,
    pipeline: Pipeline,
    param_distributions: dict,
    n_iter: int = 40,
    outer_splits: int = 5,
    inner_splits: int = 3,
    random_state: int = 42,
    n_jobs: int = -1,
):
    """对单个方案做 Nested CV，返回 outer folds 的多个评估指标（AUC, F1, Balanced Accuracy）+ 每折 best params。"""
    y = np.asarray(y)

    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=random_state)

    fold_aucs = []
    fold_f1s = []
    fold_balanced_accs = []
    fold_pr_aucs = []
    fold_precisions = []
    fold_recalls = []
    fold_best_params = []
    fold_best_estimators = []

    for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions,
            n_iter=n_iter,
            scoring="roc_auc",     # 内层调参仍以 AUC 为优化目标
            refit=True,
            cv=inner_cv,
            n_jobs=n_jobs,
            random_state=random_state,
            verbose=0,
        )
        search.fit(X_tr, y_tr)

        best_est = search.best_estimator_

        # Calculate AUC using probability scores
        score = get_score_vector(best_est, X_te)
        roc_auc = roc_auc_score(y_te, score)
        pr_auc = average_precision_score(y_te, score)

        # Calculate F1 and Balanced Accuracy using predicted labels
        y_pred = best_est.predict(X_te)
        f1 = f1_score(y_te, y_pred)
        balanced_acc = balanced_accuracy_score(y_te, y_pred)
        precision = precision_score(y_te, y_pred)
        recall = recall_score(y_te, y_pred)

        fold_aucs.append(float(roc_auc))
        fold_f1s.append(float(f1))
        fold_balanced_accs.append(float(balanced_acc))
        fold_pr_aucs.append(float(pr_auc))
        fold_precisions.append(float(precision))
        fold_recalls.append(float(recall))
        fold_best_params.append(search.best_params_)
        fold_best_estimators.append(best_est)

    metrics = {
        "roc_auc": fold_aucs,
        "pr_auc": fold_pr_aucs,
        "f1": fold_f1s,
        "balanced_acc": fold_balanced_accs,
        "precision": fold_precisions,
        "recall": fold_recalls,
    }
    return metrics, fold_best_params, fold_best_estimators


def build_features_union(
    whitelist_cols,
    all_candidate_cols,
    num_cols,
    cat_cols,
    selector=None,               # None 表示不对 opt 做筛选
    allow_opt_empty=True,        # opt 为空时是否允许（用于 selectors["None"]）
):
    whitelist_cols = list(whitelist_cols)
    all_candidate_cols = list(all_candidate_cols)

    # 这里仍然做一次去重：opt_cols 是 “候选列中去掉白名单”
    opt_cols = [c for c in all_candidate_cols if c not in set(whitelist_cols)]

    must_pre = build_preprocessor(whitelist_cols, num_cols, cat_cols)
    must_branch = Pipeline([("prep", must_pre)])

    # opt 为空：只返回 must 分支（实现 selectors["None"] 只白名单）
    if len(opt_cols) == 0:
        if allow_opt_empty:
            return FeatureUnion([("must", must_branch)])
        raise ValueError("除了白名单之外没有可筛选特征了（opt_cols 为空）。")

    opt_pre = build_preprocessor(opt_cols, num_cols, cat_cols)

    if selector is None:
        opt_branch = Pipeline([("prep", opt_pre)])
    else:
        opt_branch = Pipeline([("prep", opt_pre), ("sel", selector)])

    return FeatureUnion([
        ("must", must_branch),
        ("opt", opt_branch),
    ])

In [3]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import (
    SelectKBest,
    SelectFromModel,
    RFE,
    mutual_info_classif,
    f_classif,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# ----------------------------
# 小工具1：安全版 SelectKBest（自动把 k 截断到可用维度）
# ----------------------------
class SafeSelectKBest(SelectKBest):
    def __init__(self, score_func=mutual_info_classif, k=10):
        super().__init__(score_func=score_func, k=k)

    def fit(self, X, y=None):
        n_features = X.shape[1]
        if isinstance(self.k, int):
            self.k = min(max(1, int(self.k)), n_features)
        return super().fit(X, y)


# ----------------------------
# 小工具2：构建“强制选 top-n_opt”式 SelectFromModel（兼容不同 sklearn 版本）
# 关键点：threshold=-inf + max_features=n_opt  => 总是选出 top n_opt
# 若你的 sklearn 不支持 max_features，会退化成普通 SelectFromModel（用 threshold）。
# ----------------------------
def make_topn_select_from_model(estimator, n_opt: int):
    try:
        return SelectFromModel(estimator=estimator, max_features=int(n_opt), threshold=-np.inf)
    except TypeError:
        # 老版本 sklearn 没有 max_features 参数
        return SelectFromModel(estimator=estimator)


# ----------------------------
# 更新版 build_experiments
# ----------------------------
def build_experiments(
    X,
    num_cols,
    cat_cols,
    whitelist_cols,
    n_opt: int = 10,
    random_state: int = 42,
):
    """
    返回 experiments: List[(name, pipeline, param_distributions)]
    - 白名单特征永远保留（must 分支无筛选）
    - 可选特征（opt 分支）用 selector 选出固定数量 n_opt（当 selector 支持 top-n 时）
    """
    all_cols = list(X.columns)
    opt_all_candidate_cols = [c for c in all_cols if c not in set(whitelist_cols)]

    # ====== 选择器（opt 分支用）======
    selectors = {}

    # Filter: Mutual Information / F-test
    selectors["MI_topn"] = SafeSelectKBest(score_func=mutual_info_classif, k=n_opt)
    selectors["F_topn"]  = SafeSelectKBest(score_func=f_classif, k=n_opt)

    # Embedded: L1 / LinearSVC-L1
    l1_lr = LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=8000,
        class_weight="balanced",
        random_state=random_state,
    )
    selectors["L1LR_topn"] = make_topn_select_from_model(l1_lr, n_opt=n_opt)

    l1_lsvc = LinearSVC(
        penalty="l1",
        dual=False,
        class_weight="balanced",
        random_state=random_state,
        max_iter=8000,
    )
    selectors["L1SVC_topn"] = make_topn_select_from_model(l1_lsvc, n_opt=n_opt)

    # Embedded: Tree importance（ExtraTrees 通常比 RF 更稳、更快拿重要性）
    et_sel = ExtraTreesClassifier(
        n_estimators=800,
        random_state=random_state,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    selectors["ET_topn"] = make_topn_select_from_model(et_sel, n_opt=n_opt)

    # Wrapper: RFE（给甲方展示“wrapper”即可；不建议加太多）
    # 注意：RFE 会显著增加训练时间；这里用 LinearSVC(L2) 做 RFE 基学习器较快
    rfe_base = LinearSVC(
        penalty="l2",
        dual=True,
        class_weight="balanced",
        random_state=random_state,
        max_iter=8000,
    )
    selectors["RFE_topn"] = RFE(estimator=rfe_base, n_features_to_select=int(n_opt), step=0.2)

    # 特殊：无筛选（All opt）
    selectors["None"] = None   # 只留白名单：通过「不给 opt 候选列」实现
    selectors["All"]  = None   # 全特征：opt 候选列=其余全部，且不筛选

    # ====== 模型库（clf）======
    models = {}

    # 线性/核
    models["LR"] = LogisticRegression(
        max_iter=8000, class_weight="balanced", solver="lbfgs", random_state=random_state
    )
    models["ENetLR"] = LogisticRegression(
        max_iter=8000, class_weight="balanced", solver="saga", penalty="elasticnet", random_state=random_state
    )
    models["SVM_RBF"] = SVC(
        kernel="rbf", class_weight="balanced", probability=True, random_state=random_state
    )

    # 树/Boosting
    models["RF"] = RandomForestClassifier(
        n_estimators=800, n_jobs=-1, random_state=random_state, class_weight="balanced_subsample"
    )
    models["HGB"] = HistGradientBoostingClassifier(random_state=random_state)

    # XGBoost / LightGBM / CatBoost（默认以 AUC 优化；不平衡时加 scale_pos_weight / class_weights 可在外面算）
    models["XGB"] = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        n_estimators=1200,
        random_state=random_state,
        n_jobs=-1,
        tree_method="hist",
        scale_pos_weight=5,
    )

    # ====== 为每个模型定义一份 param_distributions（RandomizedSearchCV 用）======
    # 选择器也可能需要调参：例如 L1 的 C；树选择器的深度等
    selector_param = {
        "MI_topn": {},   # k 已固定为 n_opt
        "F_topn": {},
        "L1LR_topn": {"feat__opt__sel__estimator__C": np.logspace(-3, 1, 20)},
        "L1SVC_topn": {"feat__opt__sel__estimator__C": np.logspace(-3, 1, 20)},
        "ET_topn": {
            "feat__opt__sel__estimator__max_depth": [None, 3, 5, 8, 12],
            "feat__opt__sel__estimator__min_samples_leaf": [1, 2, 5, 10],
        },
        "RFE_topn": {
            # RFE 本身不太需要调参；可调基学习器 C
            "feat__opt__sel__estimator__C": np.logspace(-3, 1, 15)
        },
        "None": {},
        "All": {},
    }

    model_param = {
        "LR": {"clf__C": np.logspace(-3, 2, 30)},
        "ENetLR": {
            "clf__C": np.logspace(-3, 2, 25),
            "clf__l1_ratio": np.linspace(0.1, 0.9, 9),
        },
        "SVM_RBF": {
            "clf__C": np.logspace(-2, 2, 20),
            "clf__gamma": np.logspace(-4, 0, 20),
        },
        "RF": {
            "clf__max_depth": [None, 3, 5, 8, 12],
            "clf__min_samples_leaf": [1, 2, 5, 10],
            "clf__max_features": ["sqrt", "log2", 0.3, 0.5, 0.8],
        },
        "HGB": {
            "clf__learning_rate": np.logspace(-3, -1, 12),
            "clf__max_depth": [2, 3, 4, None],
            "clf__max_leaf_nodes": [15, 31, 63],
            "clf__min_samples_leaf": [10, 20, 30, 50],
        },
        "XGB": {
            "clf__learning_rate": np.logspace(-3, -1, 12),
            "clf__max_depth": [2, 3, 4, 5, 6],
            "clf__subsample": np.linspace(0.6, 1.0, 5),
            "clf__colsample_bytree": np.linspace(0.6, 1.0, 5),
            "clf__reg_lambda": np.logspace(-2, 2, 10),
            "clf__min_child_weight": [1, 2, 5, 10],
        },
    }

    # ====== 组合策略：更多 selector × model 组合 ======
    # 你可以按需要“扩大全组合”或“只挑代表性组合”
    combo_plan = [
        # Filter × 多模型
        ("MI_topn", ["LR", "ENetLR", "SVM_RBF", "RF", "HGB", "XGB"]),
        ("F_topn",  ["LR", "ENetLR", "SVM_RBF", "RF", "HGB", "XGB"]),

        # Embedded(L1) × 线性/核/Boosting
        ("L1LR_topn",  ["LR", "ENetLR", "SVM_RBF"]),
        ("L1SVC_topn", ["LR", "SVM_RBF"]),

        # Tree importance selector × 树/boosting 为主（也可配 LR，但通常意义不大）
        ("ET_topn", ["RF", "HGB", "XGB"]),

        # Wrapper（展示用）
        ("RFE_topn", ["LR", "SVM_RBF"]),

        # 无筛选基线（All features）
        ("All", ["LR", "ENetLR", "SVM_RBF", "RF", "HGB", "XGB"]),

        # 无筛选基线（only whitelist）
        ("None", ["LR", "ENetLR", "SVM_RBF", "RF", "HGB", "XGB"]),

    ]

    experiments = []

    for sel_name, model_list in combo_plan:
        selector = selectors[sel_name]
        if sel_name == "None":
            all_candidate_cols_for_opt = []  # opt 分支为空 => 最终只剩 must=白名单
        else:
            all_candidate_cols_for_opt = opt_all_candidate_cols  # All/各种 topn 都从这里选

        for model_name in model_list:
            clf = models[model_name]

            # 你的 build_features_union() 在外部已定义：must分支固定保留，opt分支可筛选
            features = build_features_union(
                whitelist_cols=whitelist_cols,
                all_candidate_cols=all_candidate_cols_for_opt,
                num_cols=num_cols,
                cat_cols=cat_cols,
                selector=selector
            )

            pipe = Pipeline([("feat", features), ("clf", clf)])

            # 合并 selector + model 的参数空间
            params = {}
            params.update(selector_param.get(sel_name, {}))
            params.update(model_param.get(model_name, {}))

            exp_name = f"{sel_name}(n_opt={n_opt})+{model_name}"
            experiments.append((exp_name, pipe, params))

    return experiments

In [4]:
import warnings


def run_all(
    X: pd.DataFrame,
    y,
    num_cols,
    cat_cols,
    whitelist_cols,
    n_iter: int = 40,
    n_opt: int = 10,
    outer_splits: int = 5,
    inner_splits: int = 3,
    random_state: int = 42,
    n_jobs: int = -1,
):
    # 基本检查
    missing = [c for c in whitelist_cols if c not in X.columns]
    if missing:
        raise ValueError(f"白名单特征不在 X.columns 中：{missing}")

    experiments = build_experiments(X, num_cols, cat_cols, whitelist_cols, n_opt=n_opt, random_state=random_state)

    rows = []
    all_details = {}  # 你如果想看每折 best_params，可以从这里拿

    for name, pipe, params in tqdm(experiments):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            metrics, best_params, best_estimators = nested_cv_evaluate(
                X, y,
                pipeline=pipe,
                param_distributions=params,
                n_iter=n_iter,
                outer_splits=outer_splits,
                inner_splits=inner_splits,
                random_state=random_state,
                n_jobs=n_jobs,
            )

        aucs = metrics["roc_auc"]
        f1s = metrics["f1"]
        balanced_accs = metrics["balanced_acc"]
        pr_aucs = metrics["pr_auc"]
        precisions = metrics["precision"]
        recalls = metrics["recall"]

        rows.append({
            "method": name,
            "auc_mean": float(np.mean(aucs)),
            "auc_sd": float(np.std(aucs, ddof=1)) if len(aucs) > 1 else 0.0,
            "f1_mean": float(np.mean(f1s)),
            "f1_sd": float(np.std(f1s, ddof=1)) if len(f1s) > 1 else 0.0,
            "balanced_acc_mean": float(np.mean(balanced_accs)),
            "balanced_acc_sd": float(np.std(balanced_accs, ddof=1)) if len(balanced_accs) > 1 else 0.0,
            "pr_auc_mean": float(np.mean(pr_aucs)),
            "pr_auc_sd": float(np.std(pr_aucs, ddof=1)) if len(pr_aucs) > 1 else 0.0,
            "precision_mean": float(np.mean(precisions)),
            "precision_sd": float(np.std(precisions, ddof=1)) if len(precisions) > 1 else 0.0,
            "recall_mean": float(np.mean(recalls)),
            "recall_sd": float(np.std(recalls, ddof=1)) if len(recalls) > 1 else 0.0,
            "auc_folds": aucs,
            "f1_folds": f1s,
            "balanced_acc_folds": balanced_accs,
            "pr_auc_folds": pr_aucs,
            "precision_folds": precisions,
            "recall_folds": recalls,
        })
        all_details[name] = {
            "metrics_per_fold": metrics,
            "best_params_per_fold": best_params,
            "best_estimators": best_estimators,  # 如果你要做特征稳定性分析，可用这个
        }

    res = pd.DataFrame(rows).sort_values("auc_mean", ascending=False).reset_index(drop=True)
    return res, all_details

In [5]:
train = pd.read_csv('../data/train.csv')
X = train.drop(columns=['Endpoint'])
y = train['Endpoint']
num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
whitelist_cols = ['A1BG', 'A1AT', 'A2MG', 'APOB']
res, details = run_all(X, y, num_cols, cat_cols, whitelist_cols, n_opt=6, n_iter=50)

 50%|█████     | 17/34 [05:37<02:06,  7.45s/it]/Users/fubin/miniforge3/envs/ml/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/fubin/miniforge3/envs/ml/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/fubin/miniforge3/envs/ml/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(

In [7]:
res

,method,auc_mean,auc_sd,f1_mean,f1_sd,balanced_acc_mean,balanced_acc_sd,pr_auc_mean,pr_auc_sd,precision_mean,precision_sd,recall_mean,recall_sd,auc_folds,f1_folds,balanced_acc_folds,pr_auc_folds,precision_folds,recall_folds
0,MI_topn(n_opt=6)+RF,0.794893,0.084085,0.276107,0.225938,0.572984,0.109435,0.517042,0.155717,0.344444,0.255797,0.246429,0.213690,"[0.8083333333333333, 0.7095238095238096, 0.719...","[0.18181818181818182, 0.0, 0.25, 0.61538461538...","[0.5291666666666667, 0.4666666666666667, 0.526...","[0.5013304093567252, 0.36944444444444446, 0.41...","[0.3333333333333333, 0.0, 0.2222222222222222, ...","[0.125, 0.0, 0.2857142857142857, 0.57142857142..."
1,ET_topn(n_opt=6)+XGB,0.789762,0.105549,0.467930,0.077019,0.680998,0.059672,0.511235,0.194352,0.380689,0.069064,0.617857,0.127275,"[0.875, 0.6714285714285714, 0.7285714285714285...","[0.5, 0.5, 0.3333333333333333, 0.5263157894736...","[0.6958333333333333, 0.7023809523809523, 0.580...","[0.6032738095238096, 0.32489207804333853, 0.35...","[0.4166666666666667, 0.4444444444444444, 0.272...","[0.625, 0.5714285714285714, 0.4285714285714285..."
2,All(n_opt=6)+RF,0.787685,0.094322,0.177143,0.176589,0.539528,0.060857,0.487465,0.156029,0.290476,0.297571,0.139286,0.143080,"[0.8, 0.7666666666666667, 0.7047619047619047, ...","[0.0, 0.0, 0.2857142857142857, 0.4, 0.2]","[0.5, 0.4666666666666667, 0.5595238095238095, ...","[0.45530861801242234, 0.4166118060127277, 0.32...","[0.0, 0.0, 0.2857142857142857, 0.6666666666666...","[0.0, 0.0, 0.2857142857142857, 0.2857142857142..."
3,L1SVC_topn(n_opt=6)+LR,0.779610,0.050075,0.493890,0.099751,0.699680,0.078085,0.536580,0.073494,0.414343,0.086776,0.621429,0.146385,"[0.7208333333333333, 0.7904761904761906, 0.766...","[0.47058823529411764, 0.5882352941176471, 0.33...","[0.6666666666666667, 0.7738095238095238, 0.580...","[0.524563127812901, 0.5892063492063492, 0.4416...","[0.4444444444444444, 0.5, 0.2727272727272727, ...","[0.5, 0.7142857142857143, 0.42857142857142855,..."
4,F_topn(n_opt=6)+LR,0.778186,0.039079,0.544354,0.092429,0.733247,0.058546,0.529097,0.072369,0.466697,0.124694,0.675000,0.074059,"[0.7833333333333333, 0.8, 0.7333333333333334, ...","[0.625, 0.625, 0.4, 0.5263157894736842, 0.5454...","[0.7625, 0.7904761904761906, 0.635714285714285...","[0.5439980158730159, 0.5866141328326202, 0.436...","[0.625, 0.5555555555555556, 0.3076923076923077...","[0.625, 0.7142857142857143, 0.5714285714285714..."
5,F_topn(n_opt=6)+XGB,0.773690,0.074844,0.438290,0.088590,0.666741,0.072237,0.515552,0.170693,0.344034,0.044688,0.664286,0.246597,"[0.7708333333333333, 0.6904761904761905, 0.761...","[0.5384615384615384, 0.3076923076923077, 0.421...","[0.7541666666666667, 0.5761904761904761, 0.652...","[0.5194765406162465, 0.34173899444835404, 0.35...","[0.3888888888888889, 0.3333333333333333, 0.333...","[0.875, 0.2857142857142857, 0.5714285714285714..."
6,ET_topn(n_opt=6)+RF,0.771638,0.078412,0.275000,0.231241,0.576679,0.109221,0.464766,0.136758,0.309444,0.250613,0.260714,0.232005,"[0.8125, 0.6857142857142857, 0.728571428571428...","[0.3333333333333333, 0.16666666666666666, 0.25...","[0.5916666666666667, 0.5047619047619047, 0.526...","[0.48353174603174603, 0.339868068439497, 0.350...","[0.5, 0.2, 0.2222222222222222, 0.0, 0.625]","[0.25, 0.14285714285714285, 0.2857142857142857..."
7,F_topn(n_opt=6)+ENetLR,0.771289,0.047151,0.508619,0.074889,0.704331,0.042382,0.541865,0.077658,0.452731,0.113567,0.617857,0.127275,"[0.7833333333333333, 0.8142857142857143, 0.733...","[0.625, 0.42857142857142855, 0.476190476190476...","[0.7625, 0.6476190476190476, 0.707142857142857...","[0.5439980158730159, 0.6035409035409036, 0.438...","[0.625, 0.42857142857142855, 0.357142857142857...","[0.625, 0.42857142857142855, 0.714285714285714..."
8,F_topn(n_opt=6)+RF,0.767475,0.057884,0.313997,0.181367,0.603103,0.065472,0.415978,0.067460,0.391775,0.246430,0.300000,0.215177,"[0.7708333333333333, 0.7380952380952381, 0.733...","[0.36363636363636365, 0.42857142857142855, 0.4...","[0.6083333333333334,

In [8]:
res.to_csv("../results/data/benchmark.csv", index=False)